# Train Test Creator

Reads the pre-selected **`<target>__lb<L>__final`** VIEW from `unified_schema_<ticker>`
and builds windowed train/val/test tensors. Each **sample** is a
`(LOOKBACK_DAY, n_features)` window of all view features ending on day *t*, with
**label = `target` at day *t*** (the 5-day-ahead return `return_5day`).

Feature selection + TA tuning are already done upstream in
`unified_schema_creator.ipynb` (the view is pre-selected), so this notebook only:
**clean → chronological split → scale (train-fit) → window → save**.

## Import Libraries

In [1]:
import json
import os
import sys

import joblib
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, os.path.abspath(".."))

from dtos.tabular_database_driver_dtos.postgre_sql_connection_dto import PostgreSQLConnectionDto
from logger.logger import Logger
from tabular_database_driver.postgre_sql_driver import PostgreSQLDriver
from utils.constants import DATABASE_MAIN_V2

load_dotenv()

True

## Parameters

In [2]:
TICKER = "vcb"
TARGET = "return_5day"       # the pool__targets column selected upstream
LOOKBACK_DAY = 20            # window length (must match the __final view's lb)
TARGET_HORIZON = 5           # target looks 5 trading days into the future

SCHEMA = f"unified_schema_{TICKER.lower()}"
VIEW = f"{TARGET}__lb{LOOKBACK_DAY}__final"

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = round(1 - TRAIN_RATIO - VAL_RATIO, 4)  # 0.15 (implicit)

DATE_COL = "date"
TARGET_COLUMN = "target"
SCALER_TAG = "std"           # std = StandardScaler
SCALE_TARGET = True          # standardize target (scaler saved for inverse-transform)
RANDOM_STATE = 42

PG_HOST = os.getenv("POSTGRES_HOST", "localhost")
PG_PORT = int(os.getenv("POSTGRES_PORT", 5432))
PG_USER = os.getenv("POSTGRES_USER", "postgres")
PG_PASSWORD = os.getenv("POSTGRES_PASSWORD", "")

DATASET_NAME = (
    f"{TICKER.lower()}_{TARGET}"
    f"_lb{LOOKBACK_DAY}_h{TARGET_HORIZON}_final"
    f"_tr{int(TRAIN_RATIO * 100)}_val{int(VAL_RATIO * 100)}_test{int(TEST_RATIO * 100)}"
    f"_{SCALER_TAG}"
)
OUTPUT_DIR = os.path.join("../train_test_set", DATASET_NAME)
print(f"Source view : {SCHEMA}.{VIEW}")
print(f"Dataset name: {DATASET_NAME}")

Dataset name: vcb_lb40_h5_f200_dynta_tr70_val15_test15_std


## Load Data

In [3]:
logger = Logger(file_name="../../logs/train_test_creator")

driver = PostgreSQLDriver(logger=logger)
driver.connect(
    PostgreSQLConnectionDto(
        logger=logger,
        host=PG_HOST,
        user=PG_USER,
        password=PG_PASSWORD,
        port=PG_PORT,
        database=DATABASE_MAIN_V2,
    )
)

df = driver.select(schema_name=SCHEMA, table_name=VIEW, order_by=[DATE_COL])

driver.disconnect()

print(f"Loaded {len(df)} rows, {len(df.columns)} columns from {SCHEMA}.{VIEW}")
df

Loaded 4242 rows, 153 columns from unified_schema_vcb.return_5day__lb20__final


,date,foreign_room,high,own_pct,f_buy_val,f_net_vol,vol_matched,f_sell_vol,vol_negotiated,net_change,...,close_wma_7_100_dist_abs,macd_12_26_9_hist_abs,sar_002_02_dist_abs,close_dema_50_100_dist_abs,ppo_12_26_9_strength,ultosc_7_14_28_abs,sar_002_02_004_02_dist_abs,ht_trendmode_hist_10_abs,close_tema_30_dist_abs,target
0,2009-06-30,82612532.0,9132.805,NaN,2.460000e+08,4100.0,294070.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,-0.058333
1,2009-07-01,82441852.0,9589.445,NaN,2.260555e+11,170680.0,6248390.0,3420000.0,3.0,NaN,...,NaN,NaN,380.53354,NaN,NaN,NaN,0.000000,0.000000,NaN,-0.082645
2,2009-07-02,82445952.0,9132.805,NaN,3.280495e+09,-49670.0,1515670.0,104850.0,0.0,NaN,...,NaN,NaN,761.06710,NaN,NaN,NaN,0.000000,0.000000,NaN,-0.068966
3,2009-07-03,82445952.0,8676.165,NaN,4.261395e+09,-215390.0,899720.0,290680.0,0.0,NaN,...,NaN,NaN,1065.49390,NaN,NaN,NaN,0.000000,0.000000,NaN,-0.080357
4,2009-07-06,82445952.0,8904.484,NaN,4.348780e+09,-294930.0,1571740.0,370000.0,0.0,NaN,...,NaN,NaN,621.03070,NaN,NaN,NaN,63.929634,0.000000,NaN,-0.162393
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4237,2026-06-22,816867042.0,61800.000,20.22,4.996183e+09,-617170.0,2686300.0,698484.0,0.0,-400.0,...,365.868470,113.68186,1451.45630,167.786740,0.009008,16.183224,1574.515300,0.178284,682.32910,NaN
4238,2026-06-23,818736677.0,62600.000,20.20,1.519092e+10,-148955.0,5176000.0,394290.0,0.0,0.0,...,286.786400,116.85823,1303.33970,149.089550,0.017486,20.321074,1403.339700,0.145868,569.50696,NaN
4239,2026-06-24,818942712.0,61600.000,20.20,5.317174e+09,-136600.0,2542500.0,223500.0,0.0,-300.0,...,147.438480,132.21176,1600.00000,120.451030,0.036961,20.879177,0.000000,0.119347,715.69670,NaN
4240,2026-06-25,819584516.0,61300.000,20.19,4.151386e+09,-424339.0,3093500.0,492439.0,0.0,-200.0,...,18.025460,147.70973,1800.00000,86.560380,0.065552,23.195818,0.000000,0.097647,744.52460,NaN


## Per-Stock TA Period Selection

The informative TA period differs per stock. For *this* ticker, sweep candidate period grids, regenerate the real derived features with the project's own `add_*` functions, and rank each config by regime-stable mutual information with the target (median across consecutive time blocks). Keep the top-`TA_TOP_K` configs per family. Fit on the whole series at row level (univariate, so collinear periods don't split credit).

In [4]:
# Align OHLCV with a valid target (drop the NaN-target tail) — shared by selection and generation
df_clean = df.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)
dates = pd.to_datetime(df_clean["date"])

ohlcv = df_clean[["date", "open", "high", "low", "close", VOLUME_COL]].copy()
for c in ["open", "high", "low", "close", VOLUME_COL]:
    ohlcv[c] = pd.to_numeric(ohlcv[c], errors="coerce")
y_sel = pd.to_numeric(df_clean[TARGET_COLUMN], errors="coerce").to_numpy(dtype=float)

# --- Candidate grids & registry (family -> func, fixed kwargs, list of single-config dicts) ---
vol = {"volume_col": VOLUME_COL}
def single(periods):
    return [{"n": [p]} for p in periods]

macd_grid = [{"fast": [f], "slow": [sl], "signal": [sg]}
             for f, sl, sg in product([6, 12, 19], [26, 40, 52], [9]) if f < sl]
adosc_grid = [{"fast": [f], "slow": [sl]} for f, sl in product([3, 6], [10, 20]) if f < sl]

REGISTRY = {
    "adx":   (add_adx,   {}, single([3, 7, 14, 28, 56])),
    "rsi":   (add_rsi,   {}, single([3, 7, 14, 28, 56])),
    "cci":   (add_cci,   {}, single([3, 7, 14, 28, 56])),
    "cmo":   (add_cmo,   {}, single([3, 7, 14, 28, 56])),
    "willr": (add_willr, {}, single([3, 7, 14, 28, 56])),
    "mom":   (add_mom,   {}, single([5, 10, 20, 40])),
    "roc":   (add_roc,   {}, single([5, 10, 20, 40])),
    "trix":  (add_trix,  {}, single([9, 15, 30])),
    "aroon": (add_aroon, {}, single([14, 25, 50])),
    "atr":   (add_atr,   {}, single([7, 14, 28])),
    "natr":  (add_natr,  {}, single([7, 14, 28])),
    "bop":   (add_bop,   {}, single([7, 14, 28])),
    "mfi":   (add_mfi,   vol, single([7, 14, 28])),
    "ad":    (add_ad,    vol, single([3, 10, 20])),
    "obv":   (add_obv,   vol, single([3, 10, 20])),
    "sma":      (add_sma,      {}, single([5, 10, 20, 50, 100, 200])),
    "ema":      (add_ema,      {}, single([5, 10, 20, 50, 100, 200])),
    "dema":     (add_dema,     {}, single([10, 20, 50, 100])),
    "tema":     (add_tema,     {}, single([10, 20, 50, 100])),
    "trima":    (add_trima,    {}, single([10, 20, 50, 100])),
    "wma":      (add_wma,      {}, single([7, 14, 21, 50, 100])),
    "kama":     (add_kama,     {}, single([10, 20, 50, 100, 200])),
    "t3":       (add_t3,       {}, single([5, 10, 20])),
    "midpoint": (add_midpoint, {}, single([14, 50, 100])),
    "midprice": (add_midprice, {}, single([14, 50, 100])),
    "bbands":   (add_bbands,   {}, single([10, 20, 50])),
    "macd":     (add_macd,     {}, macd_grid),
    "ppo":      (add_ppo,      {}, macd_grid),
    "adosc":    (add_adosc,    vol, adosc_grid),
}

_ORDER = ["n", "fast", "slow", "signal"]
def config_label(cfg):
    return "_".join(str(cfg[k][0]) for k in _ORDER if k in cfg)

# --- Regime-stability scoring: median MI of a column vs target across time blocks ---
ta_blocks = np.array_split(np.arange(len(ohlcv)), N_TA_BLOCKS)
def regime_mi(x):
    mis = []
    for idx in ta_blocks:
        xb, yb = x[idx], y_sel[idx]
        m = np.isfinite(xb) & np.isfinite(yb)
        if m.sum() < 30:
            continue
        xv, yv = xb[m], yb[m]
        if np.unique(xv).size < 3:
            mis.append(0.0)
            continue
        mis.append(mutual_info_regression(xv.reshape(-1, 1), yv, random_state=RANDOM_STATE)[0])
    return np.median(mis) if mis else np.nan

# --- Score every config, keep the best column's MI per config ---
ohlcv_cols = set(ohlcv.columns)
config_scores = []  # (family, cfg, label, best_mi)
for family, (func, fixed, configs) in REGISTRY.items():
    for cfg in configs:
        try:
            out = func(ohlcv, **fixed, **cfg)
        except Exception as e:
            print(f"  [skip] {family} {config_label(cfg)}: {type(e).__name__}: {e}")
            continue
        best = 0.0
        for col in out.columns:
            if col in ohlcv_cols:
                continue
            x = pd.to_numeric(out[col], errors="coerce").to_numpy(dtype=float)
            if not np.isfinite(x).any() or np.nanstd(x) == 0:
                continue
            mi = regime_mi(x)
            if np.isfinite(mi):
                best = max(best, mi)
        config_scores.append((family, cfg, config_label(cfg), best))
    print(f"  scored {family}")

# --- Keep top-K configs per family ---
selected_configs = {}
recommended_periods = {}
for family in REGISTRY:
    ranked = sorted([c for c in config_scores if c[0] == family], key=lambda r: r[3], reverse=True)
    top = ranked[:TA_TOP_K]
    selected_configs[family] = [r[1] for r in top]
    labels = [r[2] for r in top]
    recommended_periods[family] = sorted(int(l) for l in labels) if all("_" not in l for l in labels) else labels

print("\nPer-stock recommended periods:")
print(json.dumps(recommended_periods, indent=2))

  scored adx


  scored rsi


  scored cci


  scored cmo


  scored willr


...

  scored midprice


  scored bbands


  scored macd


  scored ppo


  scored adosc

Per-stock recommended periods:
{
  "adx": [
    14,
    28,
    56
  ],
  "rsi": [
    14,
    28,
    56
  ],
  "cci": [
    14,
    28,
    56
  ],
  "cmo": [
    14,
    28,
    56
  ],
  "willr": [
    7,
    28,
    56
  ],
  "mom": [
    5,
    20,
    40
  ],
  "roc": [
    5,
    20,
    40
  ],
  "trix": [
    9,
    15,
    30
  ],
  "aroon": [
    14,
    25,
    50
  ],
  "atr": [
    7,
    14,
    28
  ],
  "natr": [
    7,
    14,
    28
  ],
  "bop": [
    7,
    14,
    28
  ],
  "mfi": [
    7,
    14,
    28
  ],
  "ad": [
    3,
    10,
    20
  ],
  "obv": [
    3,
    10,
    20
  ],
  "sma": [
    20,
    100,
    200
  ],
  "ema": [
    5,
    20,
    100
  ],
  "dema": [
    10,
    20,
    100
  ],
  "tema": [
    10,
    50,
    100
  ],
  "trima": [
    10,
    20,
    100
  ],
  "wma": [
    14,
    21,
    100
  ],
  "kama": [
    10,
    20,
    200
  ],
  "t3": [
    5,
    10,
    20
  ],
  "midpoint": [
    14,
    50,
    100
  ],
  "m

## Generate Tuned TA & Assemble Features

Regenerate TA at the per-stock selected periods, plus the price-derived features (returns, volatility, intraday range, rolling stats). Keep the macro + datetime **context** columns from the unified table, drop its fixed-period TA. `feature_df = raw OHLCV + macro/datetime context + freshly tuned TA`.

In [5]:
# --- Generate TA at the per-stock selected configs ---
generated = {}
for family, (func, fixed, _) in REGISTRY.items():
    for cfg in selected_configs[family]:
        out = func(ohlcv, **fixed, **cfg)
        for col in out.columns:
            if col not in ohlcv_cols and col not in generated:
                generated[col] = out[col].to_numpy()

# --- Price-derived features (fixed windows, no period sweep) ---
pderiv = ohlcv.copy()
pderiv = add_returns(pderiv)
pderiv = add_intraday_range(pderiv)
pderiv = add_return_volatility(pderiv)
pderiv = add_rolling_statistics(pderiv)
for col in pderiv.columns:
    if col not in ohlcv_cols and col not in generated:
        generated[col] = pderiv[col].to_numpy()

ta_df = pd.DataFrame(generated, index=df_clean.index)

# --- Classify unified columns: keep macro + datetime context (drop fixed-period TA) ---
META = {"exchange", "ticker", "date", TARGET_COLUMN}
RAW_OHLCV = {"open", "high", "low", "close", VOLUME_COL}
MACRO_PREFIXES = ("economy_", "bonds_")
DT_TOKENS = {"day", "month", "year", "quarter", "week", "is"}

def is_context(col):
    if col in META:
        return False
    if col in RAW_OHLCV:
        return True
    if col.startswith(MACRO_PREFIXES):
        return True
    return col.split("_")[0] in DT_TOKENS

context_cols = [c for c in df_clean.columns if is_context(c)]

# --- Assemble feature matrix ---
feature_df = pd.concat(
    [df_clean[context_cols].reset_index(drop=True), ta_df.reset_index(drop=True)], axis=1
)
feature_df = feature_df.loc[:, ~feature_df.columns.duplicated()]
target_series = pd.to_numeric(df_clean[TARGET_COLUMN], errors="coerce")

n_macro = sum(c.startswith(MACRO_PREFIXES) for c in context_cols)
n_dt = sum(c.split("_")[0] in DT_TOKENS for c in context_cols)
print(f"Assembled feature_df: {feature_df.shape}")
print(f"  raw OHLCV: {len(RAW_OHLCV)} | macro: {n_macro} | datetime: {n_dt} | tuned TA + price-derived: {ta_df.shape[1]}")

Assembled feature_df: (4208, 1258)


  raw OHLCV: 5 | macro: 91 | datetime: 18 | tuned TA + price-derived: 1144


## Clean & Classify Columns

In [6]:
# Drop the NaN-target tail (last TARGET_HORIZON rows have an incomplete future)
df_clean = df.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)
dates = pd.to_datetime(df_clean[DATE_COL])
target_series = pd.to_numeric(df_clean[TARGET_COLUMN], errors="coerce")

# Features = every view column except date + target (already selected upstream)
feature_cols = [c for c in df_clean.columns if c not in (DATE_COL, TARGET_COLUMN)]
feature_df = df_clean[feature_cols].apply(pd.to_numeric, errors="coerce")

# Fill NaN (macro leading gaps, TA warmup) — forward then back fill
nan_before = int(feature_df.isna().sum().sum())
feature_df = feature_df.ffill().bfill()
nan_after = int(feature_df.isna().sum().sum())
print(f"Dropped {len(df) - len(df_clean)} NaN-target tail rows; feature NaN {nan_before} -> {nan_after}")

# Bounded columns that should NOT be scaled:
#   - cyclical encodings (sin/cos in [-1, 1])
#   - binary 0/1 flags (calendar flags, TA crossover/band flags)
cyclical_cols = [c for c in feature_df.columns if c.endswith("_sin") or c.endswith("_cos")]
binary_cols = [
    c for c in feature_df.columns
    if set(feature_df[c].dropna().unique()).issubset({0, 1, 0.0, 1.0})
]
bounded_cols = sorted(set(cyclical_cols) | set(binary_cols))
scale_cols = [c for c in feature_df.columns if c not in set(bounded_cols)]

print(f"Feature matrix: {feature_df.shape}  |  bounded (not scaled): {len(bounded_cols)}"
      f"  |  continuous (scaled): {len(scale_cols)}")
feature_df

Feature NaN filled: 62073 -> 0


Feature matrix shape: (4208, 1258)
Bounded (not scaled): 326  |  Continuous (scaled): 932


,foreign_room,high,own_pct,f_buy_val,f_net_vol,vol_matched,f_sell_vol,vol_negotiated,net_change,week_of_year,...,adosc_3_10_hist,close_wma_7_100_dist_abs,macd_12_26_9_hist_abs,sar_002_02_dist_abs,close_dema_50_100_dist_abs,ppo_12_26_9_strength,ultosc_7_14_28_abs,sar_002_02_004_02_dist_abs,ht_trendmode_hist_10_abs,close_tema_30_dist_abs
0,82612532.0,9132.805,0.00,2.460000e+08,4100.0,294070.0,0.0,0.0,-700.0,27,...,71430.53,554.37506,97.526690,380.53354,146.17116,0.500644,10.864017,0.000000,0.000000,217.146800
1,82441852.0,9589.445,0.00,2.260555e+11,170680.0,6248390.0,3420000.0,3.0,-700.0,27,...,71430.53,554.37506,97.526690,380.53354,146.17116,0.500644,10.864017,0.000000,0.000000,217.146800
2,82445952.0,9132.805,0.00,3.280495e+09,-49670.0,1515670.0,104850.0,0.0,-700.0,27,...,71430.53,554.37506,97.526690,761.06710,146.17116,0.500644,10.864017,0.000000,0.000000,217.146800
3,82445952.0,8676.165,0.00,4.261395e+09,-215390.0,899720.0,290680.0,0.0,-700.0,27,...,71430.53,554.37506,97.526690,1065.49390,146.17116,0.500644,10.864017,0.000000,0.000000,217.146800
4,82445952.0,8904.484,0.00,4.348780e+09,-294930.0,1571740.0,370000.0,0.0,-700.0,28,...,71430.53,554.37506,97.526690,621.03070,146.17116,0.500644,10.864017,63.929634,0.000000,217.146800
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4232,811618973.0,62400.000,20.28,2.228014e+10,-1517391.0,5222300.0,1877199.0,0.0,0.0,25,...,-266505.38,219.25883,206.498170,2109.14600,187.64667,0.006490,19.180073,2809.146000,0.264031,689.575300
4233,811732555.0,62300.000,20.28,2.858882e+10,-714066.0,3110000.0,1175966.0,0.0,200.0,25,...,-266505.38,294.79773,169.506150,1684.41430,187.71924,0.002802,17.820940,2524.414300,0.602156,436.592100
4234,812111195.0,62400.000,20.28,1.270252e+10,-2889388.0,6825500.0,3094137.0,0.0,400.0,25,...,-266505.38,456.84442,113.518470,1077.66130,202.53848,0.007025,12.001858,2260.061300,0.325508,60.905598
4235,813459783.0,62400.000,20.26,2.098636e+10,-2048756.0,4650700.0,2387956.0,0.0,-600.0,25,...,-266505.38,446.49080,112.118576,1487.44820,193.11488,0.001439,18.456362,2014.552200,0.266325,571.642760


## Split Indices (chronological)

In [7]:
n_rows = len(feature_df)
train_end = int(n_rows * TRAIN_RATIO)
val_end = int(n_rows * (TRAIN_RATIO + VAL_RATIO))

print(f"Total rows         : {n_rows}")
print(f"Train rows [0:{train_end}]   {dates.iloc[0].date()} -> {dates.iloc[train_end - 1].date()}")
print(f"Val   rows [{train_end}:{val_end}]   {dates.iloc[train_end].date()} -> {dates.iloc[val_end - 1].date()}")
print(f"Test  rows [{val_end}:{n_rows}]   {dates.iloc[val_end].date()} -> {dates.iloc[n_rows - 1].date()}")

Total rows         : 4237
Train rows [0:2965]   2009-06-30 -> 2021-05-14
Val   rows [2965:3601]   2021-05-17 -> 2023-11-24
Test  rows [3601:4237]   2023-11-27 -> 2026-06-19


## Feature Selection — XGBoost gain importance (fit on train rows only)

In [8]:
# Fit XGBoost on TRAIN rows only. Trees are scale-invariant, so rank on raw (filled)
# features at row level. This gain ranking is one of three metrics blended later.
X_fs = feature_df.iloc[:train_end]
y_fs = target_series.iloc[:train_end]

xgb = XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    importance_type="gain",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb.fit(X_fs, y_fs)

gain_imp = pd.Series(xgb.feature_importances_, index=feature_df.columns)
ranked = gain_imp.sort_values(ascending=False)

print(f"Fitted XGBoost on {X_fs.shape[0]} train rows, {X_fs.shape[1]} features")
print(f"Features with zero gain (never used): {(gain_imp <= 0).sum()}")
print(f"Top 10 by gain: {ranked.head(10).index.tolist()}")

Fitted XGBoost on 2945 train rows, 1258 features
Features with zero gain (never used): 390
Top 10 by gain: ['natr', 'economy_economics_vnemp', 'cmo_14_extreme', 'cmo_14_lt_minus50', 'economy_economics_vngdps', 'rsi_14_lt_30', 'atr_28_normalized', 'close_sma_100_dist', 'close_tema_100', 'return_simple']


## Importance Cross-Check (SHAP + permutation importance)

In [9]:
import shap
from sklearn.inspection import permutation_importance

# --- SHAP: mean |contribution| per feature over TRAIN rows ---
# Consistent attribution that splits credit between correlated features more fairly than raw gain.
explainer = shap.TreeExplainer(xgb)
shap_vals = explainer.shap_values(X_fs)
shap_imp = pd.Series(np.abs(shap_vals).mean(axis=0), index=feature_df.columns)

# --- Permutation importance on VAL rows (out-of-sample) ---
# Drop in R^2 when each feature is shuffled; the only metric measured off the training fit.
# n_jobs=1 to avoid joblib memory-mapping errors on Windows.
X_val_rows = feature_df.iloc[train_end:val_end]
y_val_rows = target_series.iloc[train_end:val_end]
perm = permutation_importance(
    xgb, X_val_rows, y_val_rows, n_repeats=5, random_state=RANDOM_STATE, n_jobs=1
)
perm_imp = pd.Series(perm.importances_mean, index=feature_df.columns)

# --- How much do the three metrics agree? ---
def top_n(s, n=N_FEATURES):
    return set(s.sort_values(ascending=False).head(n).index)

gain_top, shap_top, perm_top = top_n(ranked), top_n(shap_imp), top_n(perm_imp)
print(f"Top-{N_FEATURES} pairwise overlap:")
print(f"  gain & shap : {len(gain_top & shap_top)}")
print(f"  gain & perm : {len(gain_top & perm_top)}")
print(f"  shap & perm : {len(shap_top & perm_top)}")
print()
print("Top 10 by SHAP:", shap_imp.sort_values(ascending=False).head(10).index.tolist())
print("Top 10 by perm:", perm_imp.sort_values(ascending=False).head(10).index.tolist())

D:\GIT\master-thesis\mt_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Top-200 pairwise overlap:
  gain & shap : 106
  gain & perm : 54
  shap & perm : 78

Top 10 by SHAP: ['day_of_year_cos', 'economy_economics_vniryy', 'economy_economics_vnimp', 'economy_economics_vnrsyy', 'close_trima_100', 'rsi_28_signal', 'adx_56', 'trix_30_strength', 'mfi_28_signal', 'close_sma_100']
Top 10 by perm: ['day_of_year_cos', 'economy_economics_vnrsyy', 'adx_28', 'economy_economics_vncf', 'natr_28_signal', 'bonds_tvc_vn02', 'close_bb_50_bandwidth', 'economy_economics_vngdpyy', 'aroon_down_50', 'day_of_year']


## Blended Ranking & Selection (rank-percentile of gain + SHAP + perm, then redundancy prune)

In [10]:
# Blend the three metrics by averaging their rank-percentiles. Rank-percentile is robust to
# their very different scales (gain is heavy-tailed; perm can be negative for useless features).
BLEND_WEIGHTS = {"gain": 1.0, "shap": 1.0, "perm": 1.0}

metrics = pd.DataFrame(
    {
        "gain": gain_imp,
        "shap": shap_imp,
        "perm": perm_imp,
    }
)
pct_rank = metrics.rank(pct=True)  # per-metric percentile in [0, 1], higher = more important
w = pd.Series(BLEND_WEIGHTS)
blended = (pct_rank * w).sum(axis=1) / w.sum()
blended_ranked = blended.sort_values(ascending=False)

# Redundancy prune walking down the BLENDED ranking (correlation on train rows only)
corr = X_fs.corr().abs()
selected = []
for feat_name in blended_ranked.index:
    if selected:
        max_corr = corr.loc[feat_name, selected].max()
        if pd.notna(max_corr) and max_corr >= REDUNDANCY_CORR:
            continue  # redundant with an already-kept, higher-ranked feature
    selected.append(feat_name)
    if len(selected) == N_FEATURES:
        break

# Fallback: top up from the blended ranking if pruning left us short
if len(selected) < N_FEATURES:
    for feat_name in blended_ranked.index:
        if feat_name not in selected:
            selected.append(feat_name)
        if len(selected) == N_FEATURES:
            break

selected_features = selected  # ordered by blended rank (post-prune)

# Full ranking table (all features, every metric + blend + kept flag) -> feature_ranking.csv
feature_ranking = (
    metrics.assign(blended_score=blended)
    .rename(columns={
        "gain": "gain_importance",
        "shap": "shap_importance",
        "perm": "perm_importance",
    })
    .sort_values("blended_score", ascending=False)
    .reset_index()
    .rename(columns={"index": "feature"})
)
feature_ranking["selected"] = feature_ranking["feature"].isin(selected_features)

sel = set(selected_features)
print(f"Selected {len(selected_features)} / {feature_df.shape[1]} by blended rank (+ redundancy prune)")
print(f"Top 10 blended: {selected_features[:10]}")
print()
print(f"Final selection agreement with each single metric's top-{N_FEATURES}:")
print(f"  gain : {len(sel & gain_top)}")
print(f"  shap : {len(sel & shap_top)}")
print(f"  perm : {len(sel & perm_top)}")

Selected 200 / 1258 by blended rank (+ redundancy prune)
Top 10 blended: ['day_of_year_cos', 'rsi_28_signal', 'trix_30_strength', 'adx_28', 'trix_30_signal', 'volatility_21', 'economy_economics_vniryy', 'natr_28_signal', 'economy_economics_vnrsyy', 'bonds_tvc_vn02']

Final selection agreement with each single metric's top-200:
  gain : 70
  shap : 96
  perm : 139


## Normalize (fit on train rows only)

In [11]:
# Order columns [scaled continuous ... | bounded], then scale (train-fit only)
ordered_cols = scale_cols + bounded_cols
feat = feature_df[ordered_cols].copy()

# --- Feature scaler: fit on TRAIN rows only, transform the whole array ---
feature_scaler = StandardScaler()
feature_scaler.fit(feat.iloc[:train_end][scale_cols].values)
feat[scale_cols] = feature_scaler.transform(feat[scale_cols].values)

# --- Target: optionally standardize (fit on TRAIN rows only) ---
if SCALE_TARGET:
    target_scaler = StandardScaler()
    target_scaler.fit(target_series.iloc[:train_end].values.reshape(-1, 1))
    y_full = target_scaler.transform(target_series.values.reshape(-1, 1)).ravel()
else:
    target_scaler = None
    y_full = target_series.values

X_arr = feat.values.astype(np.float32)
y_arr = y_full.astype(np.float32)

print(f"Scaled feature array: {X_arr.shape}  ({len(scale_cols)} scaled, {len(bounded_cols)} bounded)")
if SCALE_TARGET:
    print(f"Target scaled  mean~0: {y_arr[:train_end].mean():.4f}  std~1: {y_arr[:train_end].std():.4f}")

Scaled feature array: (4237, 151)  (142 scaled, 9 bounded)
Target scaled  mean~0: 0.0000  std~1: 1.0000


## Create Windows (3D tensors)

In [12]:
def make_windows(X, y, start, end):
    """Sliding windows of length LOOKBACK_DAY; target is the value at the window's last day.

    Each split starts LOOKBACK_DAY-1 rows early so its first window is complete
    without borrowing target rows from the previous split (no leakage —
    features are scaled with train-only statistics, targets only look forward).
    """
    xs, ys = [], []
    for i in range(start, end - LOOKBACK_DAY + 1):
        xs.append(X[i : i + LOOKBACK_DAY])
        ys.append(y[i + LOOKBACK_DAY - 1])
    return np.stack(xs).astype(np.float32), np.array(ys, dtype=np.float32)

X_train, y_train = make_windows(X_arr, y_arr, 0, train_end)
X_val,   y_val   = make_windows(X_arr, y_arr, train_end - (LOOKBACK_DAY - 1), val_end)
X_test,  y_test  = make_windows(X_arr, y_arr, val_end - (LOOKBACK_DAY - 1), n_rows)

print(f"X_train : {X_train.shape}  y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}  y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape}  y_test  : {y_test.shape}")

X_train : (2906, 40, 200)  y_train : (2906,)
X_val   : (631, 40, 200)  y_val   : (631,)
X_test  : (632, 40, 200)  y_test  : (632,)


## Save Model-Ready Datasets

In [13]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Tensors
np.save(os.path.join(OUTPUT_DIR, "X_train.npy"), X_train)
np.save(os.path.join(OUTPUT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(OUTPUT_DIR, "X_val.npy"), X_val)
np.save(os.path.join(OUTPUT_DIR, "y_val.npy"), y_val)
np.save(os.path.join(OUTPUT_DIR, "X_test.npy"), X_test)
np.save(os.path.join(OUTPUT_DIR, "y_test.npy"), y_test)

# Scalers (joblib handles sklearn objects well)
joblib.dump(feature_scaler, os.path.join(OUTPUT_DIR, "feature_scaler.pkl"))
if target_scaler is not None:
    joblib.dump(target_scaler, os.path.join(OUTPUT_DIR, "target_scaler.pkl"))

# Metadata
metadata = {
    "dataset_name": DATASET_NAME,
    "ticker": TICKER.lower(),
    "schema": SCHEMA,
    "source_view": VIEW,
    "lookback_day": LOOKBACK_DAY,
    "target_horizon": TARGET_HORIZON,
    "target": TARGET,
    "target_column": TARGET_COLUMN,
    "n_features": len(ordered_cols),
    "feature_columns": ordered_cols,
    "scaled_columns": scale_cols,
    "bounded_columns": bounded_cols,
    "feature_selection": (
        "done upstream in unified_schema_creator.ipynb; the "
        f"{VIEW} view is already the selected feature set"
    ),
    "split_ratios": {"train": TRAIN_RATIO, "val": VAL_RATIO, "test": TEST_RATIO},
    "shapes": {
        "X_train": list(X_train.shape), "y_train": list(y_train.shape),
        "X_val": list(X_val.shape),     "y_val": list(y_val.shape),
        "X_test": list(X_test.shape),   "y_test": list(y_test.shape),
    },
    "date_ranges": {
        "train": [str(dates.iloc[0].date()), str(dates.iloc[train_end - 1].date())],
        "val":   [str(dates.iloc[train_end].date()), str(dates.iloc[val_end - 1].date())],
        "test":  [str(dates.iloc[val_end].date()), str(dates.iloc[n_rows - 1].date())],
    },
    "scaler": {
        "feature": "StandardScaler",
        "target": "StandardScaler" if SCALE_TARGET else None,
    },
    "nan_handling": {"target": "dropped tail NaN rows", "features": "ffill+bfill"},
    "random_state": RANDOM_STATE,
    "created_at": pd.Timestamp.now().strftime("%Y-%m-%d"),
}
with open(os.path.join(OUTPUT_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved to {OUTPUT_DIR}")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1e6
    print(f"  {fname:24s} {size:8.2f} MB")

Saved to ../train_test_set\vcb_lb40_h5_f200_dynta_tr70_val15_test15_std
  X_test.npy                  20.22 MB
  X_train.npy                 92.99 MB
  X_val.npy                   20.19 MB
  feature_ranking.csv          0.10 MB
  feature_scaler.pkl           0.01 MB
  metadata.json                0.02 MB
  target_scaler.pkl            0.00 MB
  y_test.npy                   0.00 MB
  y_train.npy                  0.01 MB
  y_val.npy                    0.00 MB
